# LLM Coding Notebook — Content Analysis

**Project:** Gates Foundation / Norman Lear Center — Manfluencer Study  
**Purpose:** Code 200 Nigeria + 200 Kenya content items using GPT-4o-mini, replicating the human content codebook (Q1–Q18b).  
**Model:** gpt-4o-mini, SEED=42, async concurrency=8  
**Caching:** Parquet cache in `ROOT/temp/` — re-runs skip already-coded rows.  
**Output:** `ROOT/Codebooks/LLM Codebook/LLM Coding - Content Analysis.xlsx`


In [ ]:
# ── Cell 1: Imports & Configuration ───────────────────────────────────────────
import os
import json
import asyncio
import time
import re
from pathlib import Path
from datetime import datetime

import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openai import AsyncOpenAI

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT = Path("/Users/sushildalavi/Desktop/NLC/Gates-Manfluencer-Project")
CONTENT_MASTER = ROOT / "Codebooks/Human Codebooks/Master Human Content Codebook.xlsx"
OUTPUT_PATH    = ROOT / "Codebooks/LLM Codebook/LLM Coding - Content Analysis.xlsx"
CACHE_DIR      = ROOT / "temp"
CACHE_PATH     = CACHE_DIR / "llm_content_cache.parquet"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Model config ───────────────────────────────────────────────────────────────
MODEL      = "gpt-4o-mini"
SEED       = 42
CONCURRENCY = 8
MAX_RETRIES = 3
BATCH_SAVE  = 50   # save cache every N rows

# ── OpenAI client ─────────────────────────────────────────────────────────────
client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])

print(f"ROOT:          {ROOT}")
print(f"Content master: {CONTENT_MASTER.exists()}")
print(f"Output path:   {OUTPUT_PATH}")
print(f"Cache path:    {CACHE_PATH}")
print(f"Model:         {MODEL}, SEED={SEED}, concurrency={CONCURRENCY}")


In [ ]:
# ── Cell 2: Output column definitions (EXACT header names, 33 cols) ────────────

OUTPUT_COLUMNS = [
    "Content ID",
    "Context",
    "Content Text / Description",
    "Q1. Attention-getter (Yes/No)",
    "Q1a. Attention-getting strategies",
    "Q1b. Other strategy",
    "Q2. Primary topic(s)",
    "Q2a. Other topic",
    "Q3. Type of content",
    "Q3a. Other content type",
    "Q4. Addresses masculinity / gender norms",
    "Q5. Type of masculinity / gender norms",
    "Q6. Addresses what men should do",
    "Q7. What men do or need to do",
    "Q7a. Other directive",
    "Q8. Problem identified",
    "Q8a. Other problem",
    "Q9. Solution proposed",
    "Q9a. Other solution",
    "Q10. Communication mode",
    "Q10a. Other communication mode",
    "Q11. Audience needs",
    "Q12. How claims are supported",
    "Q12a. Other claim support",
    "Q13. How claims are justified",
    "Q13a. Other justification",
    "Q14. Sentiment toward men",
    "Q15. Sentiment toward women",
    "Q16. Sentiment toward traditional gender norms",
    "Q17. Fear or threat used",
    "Q18. Calls to action present",
    "Q18a. Types of calls to action",
    "Q18b. Other call to action",
]

assert len(OUTPUT_COLUMNS) == 33, f"Expected 33 cols, got {len(OUTPUT_COLUMNS)}"
print(f"Output columns defined: {len(OUTPUT_COLUMNS)}")


In [ ]:
# ── Cell 3: Load input data ────────────────────────────────────────────────────

def load_content_sheet(sheet_name: str) -> pd.DataFrame:
    """Load a country sheet from the master content codebook.
    Returns only the columns we feed to the LLM: Content ID, Context, Content Text / Description.
    """
    df = pd.read_excel(CONTENT_MASTER, sheet_name=sheet_name, dtype=str)
    df = df.fillna("")
    # Keep only input columns (positions 0, 3, 4 → 0-indexed)
    needed = ["Content ID", "Context", "Content Text / Description"]
    for col in needed:
        assert col in df.columns, f"Missing column: {col}"
    return df[needed].copy()

nigeria_df = load_content_sheet("Nigeria \u2014 All Items")
kenya_df   = load_content_sheet("Kenya \u2014 All Items")

assert len(nigeria_df) == 200, f"Nigeria: expected 200 rows, got {len(nigeria_df)}"
assert len(kenya_df)   == 200, f"Kenya: expected 200 rows, got {len(kenya_df)}"

print(f"Nigeria rows: {len(nigeria_df)}")
print(f"Kenya rows:   {len(kenya_df)}")

assert nigeria_df["Content ID"].astype(str).nunique() == len(nigeria_df), "Nigeria has duplicate Content IDs"
assert kenya_df["Content ID"].astype(str).nunique() == len(kenya_df), "Kenya has duplicate Content IDs"

print()
print("Nigeria sample:")
print(nigeria_df.head(2).to_string())


In [ ]:
# ── Cell 4: Cache helpers ──────────────────────────────────────────────────────

def load_cache() -> pd.DataFrame:
    """Load existing cache or return empty DataFrame."""
    if CACHE_PATH.exists():
        df = pd.read_parquet(CACHE_PATH)
        print(f"Loaded cache: {len(df)} rows from {CACHE_PATH}")
        return df
    print("No cache found — starting fresh.")
    return pd.DataFrame()

def save_cache(cache_df: pd.DataFrame):
    """Persist cache to parquet."""
    cache_df.to_parquet(CACHE_PATH, index=False)

def cache_key(content_id: str, country: str) -> str:
    return f"{country}::{content_id}::{MODEL}"

cache_df = load_cache()
cached_keys = set(cache_df["_cache_key"].tolist()) if "_cache_key" in cache_df.columns else set()
print(f"Cached keys: {len(cached_keys)}")


In [ ]:
# ── Cell 5: System prompt and coding prompt builder ────────────────────────────

THEME_TAXONOMY = """
=== RESEARCH THEME TAXONOMY (background context — not a question to answer) ===
This taxonomy describes the 13 substantive masculinity themes found across the dataset.
Use it to understand domain context while answering Q1–Q18b.

1. Authority and Submission — explicit hierarchy, headship, obedience, control, surrendering independence,
   surname, command, male leadership over women/family
2. Male Victimhood — men/boys framed as harmed, exploited, ignored, falsely accused, abused, disadvantaged,
   or punished by women, courts, culture, or institutions. Central claim: men are harmed.
3. Gender Grievance — generalized distrust, resentment, or hostility toward women, feminists, modern women,
   equality, or gender change. Central claim: women/feminism are corrupt or dangerous.
4. Sexual Morality — cheating, body count, abortion, pornography, sexual shame, sexual respectability,
   body policing, purity, double standards
5. Relationship Tactics — tactical dating/relationship advice: scarcity, pursuit, withdrawal, dominance,
   masculine frame
6. Provider and Status — money, career, income, provision, status, economic value as proof of manhood
   or relationship worth
7. Male Accountability — men must change, hold other men accountable, challenge harmful norms, refuse deflection
8. Egalitarian Partnership — mutual respect, shared parenting, allyship, listening, reciprocity,
   non-hierarchical partnership
9. Gender-Based Violence and Consent — rape, consent, sexual assault, abuse, victim-blaming, victim stigma,
   survivor support, justice, child abuse, FGM, femicide, prevention
10. Trauma and Mental Health — trauma, depression, grief, shame, emotional suppression, healing, therapy,
    vulnerability, help-seeking
11. Self-Discipline and Self-Mastery — discipline, restraint, fitness, hustle, personal growth,
    emotional control, self-improvement as masculine development (code regardless of framing)
12. Marriage and Family — marriage, divorce, fatherhood, motherhood, parenting, family stability
    (if hierarchy/headship/submission is dominant → use Authority and Submission instead)
13. Faith and Moral Repair — God, prayer, scripture, sin, testimony, repentance, spiritual healing
    tied to masculinity/gender/family/relationships/accountability
Unclear — uncodable, off-topic, low-signal, no clear gender/masculinity theme.

Tie-breaker rules:
- "Submit/obey/surrender/head of household" dominates → Authority and Submission (not Marriage and Family)
- Dating strategy/scarcity/frame/withdrawal → Relationship Tactics (not Provider and Status)
- Men harmed/ignored/exploited → Male Victimhood (not Gender Grievance)
- Women/feminism/equality framed as threat → Gender Grievance (not Male Victimhood)
- False accusations in GBV context → GBV and Consent; as general grievance → Male Victimhood or Gender Grievance
- Faith as primary moral frame for gender/healing/marriage → Faith and Moral Repair
- Self-discipline even if used regressively → Self-Discipline and Self-Mastery
"""

NIGERIA_CONTEXT = (
    "Note on Nigeria context: Pidgin/Yoruba/Igbo/Hausa terms may appear. Faith framing is common "
    "(Christian + Islamic). Creators include: Agba John Doe, Banky Wellington, Deyemi Okanlawon, "
    "Ebuka Obi-Uchendu, Shola, Wizarab."
)

KENYA_CONTEXT = (
    "Note on Kenya context: Sheng/Swahili terms may appear. Andrew Kibe/Jagero is a regressive "
    "manosphere influencer. Rixpoet/Onyango is a progressive mental health advocate. Philip Karanja "
    "focuses on GBV/femicide/FGM. Eddy Kimani is a progressive mental health/fatherhood speaker."
)

SYSTEM_PROMPT = """You are a precise qualitative research assistant for a study on masculinity content 
in Nigeria and Kenya. You will be given content from social media influencers and must apply a 
structured content analysis codebook.

Rules:
- Answer ONLY using the exact options listed for each question. Do not invent options.
- For multi-select questions, return a JSON array of strings (can be empty only where explicitly allowed).
- For single-select questions, return a single string.
- For open-text questions, return a string (can be empty string if not applicable).
- Return ONLY valid JSON — no markdown, no code blocks, no commentary.
- If selecting "Other" anywhere, provide specific detail in the matching open-text field (minimum 5 words); never leave it blank.
- Be thorough and analytical. Do not default to "No" or easy answers."""


def build_content_prompt(content_id: str, country: str, context: str, text: str) -> str:
    country_note = NIGERIA_CONTEXT if country == "Nigeria" else KENYA_CONTEXT
    return f"""You are coding a piece of content from a {country} influencer for a research study on masculinity.

{country_note}

{THEME_TAXONOMY}

=== CONTENT TO CODE ===
Content ID: {content_id}
Country: {country}
Context (platform / creator info): {context if context else '(not provided)'}
Content Text / Description:
{text}

=== CODEBOOK: Answer each question below ===

Q1. Is this an attention-getter? (does it use hooks/openers to capture attention)
  Options [single]: Yes | No

Q1a. Attention-getting strategies used (ONLY if Q1=Yes; else return empty list)
  Options [multi-select]: Compelling question | Use of all CAPS | Humor or sarcasm | 
  Shares something violent or gross | Shares something sexual | Shares something surprising | 
  Uses a news headline or social media trend as opener | Interesting visual or meme | Other

Q1b. If Q1a includes "Other", describe the other strategy. Else empty string.

Q2. Primary topic(s) of the content
  Options [multi-select]: Dating/marriage | Friends/socializing | Family/children | Money/status | 
  Fitness/self-improvement | Mental health | Gender issues, e.g. equality | 
  Social issues, e.g. corruption | Religion/morality | Gaming/technology | Other

Q2a. If Q2 includes "Other", describe the other topic. Else empty string.

Q3. Type of content
  Options [single]: Interview/conversational content | Motivational/self-help content | 
  Commentary/reaction content | Other

Q3a. If Q3="Other", describe the content type. Else empty string.

Q4. Does the content address masculinity or gender norms?
  Options [single]: Yes, explicitly | Yes, implicitly | No

Q5. Type of masculinity / gender norms expressed
  Options [single]: More regressive/traditional/restrictive | More progressive/equitable/expansive | 
  Mixed/unclear | Does not address masculinity or gender norms
  (If Q4=No, answer "Does not address masculinity or gender norms")

Q6. Does the content address what men should do?
  Options [single]: Yes | No | Unclear | Not applicable
  (If Q4=No, answer "Not applicable")

Q7. What men are told they do or need to do (ONLY if Q6=Yes; else return ["Not applicable"])
  Options [multi-select]: Men need to dominate/lead | Men need to provide/succeed | 
  Men are disadvantaged/victims | Men need to improve themselves | Men need to be fully self-reliant | 
  Men need to be emotionally open | Men need to not show emotions | Men need to be equal partners | 
  Mixed/unclear | Other | Not applicable

Q7a. If Q7 includes "Other", describe the other directive. Else empty string.

Q8. Problem identified in the content
  Options [multi-select]: Kenyan or Nigerian political/social problems | 
  Global political/social/cultural problems | Western political/social influence | 
  Women/feminism | Men's behavior | Economic/status pressure | 
  Mental health/emotional struggle | No clear problem is identified | Other

Q8a. If Q8 includes "Other", describe the other problem. Else empty string.

Q9. Solution proposed in the content
  Options [multi-select]: Social or political change | Assert dominance/control | 
  More wealth/status | More self-discipline/fitness | More emotional growth/healing | 
  More equality/respect for men | More equality/respect for women | 
  Building community | No clear solution | Other

Q9a. If Q9 includes "Other", describe the other solution. Else empty string.

Q10. Communication mode used
  Options [multi-select]: Advice/instruction | Personal story | Commentary/opinion | 
  Debate/argument | Humor/satire | Motivational speech | News/telling facts | Other

Q10a. If Q10 includes "Other", describe the other communication mode. Else empty string.

Q11. Audience needs this content serves
  Options [multi-select]: Entertainment/escapism | Information seeking | 
  Connection/social interaction | Self expression/identity construction | 
  Status seeking | Documentation of events | None of these apply

Q12. How claims are supported in the content
  Options [single]: Generalizations about men/women | Personal experience | 
  Stories about men/women | Cultural/social observations | Facts/statistics | 
  Moral/religious claims | Mixed | No support | Other

Q12a. If Q12="Other", describe the other claim support. Else empty string.

Q13. How claims are justified
  Options [single]: No justification | Anecdotal examples | Presented as common sense | 
  References data | References religion/tradition | 
  References external sources, such as other influencers | Other

Q13a. If Q13="Other", describe the other justification. Else empty string.

Q14. Sentiment toward men expressed in the content
  Options [single]: Negative | Positive | Mixed | Neutral | Unclear | Not mentioned

Q15. Sentiment toward women expressed in the content
  Options [single]: Negative | Positive | Mixed | Neutral | Unclear | Not mentioned

Q16. Sentiment toward traditional gender norms expressed in the content
  Options [single]: Negative | Positive | Mixed | Neutral | Unclear | Not mentioned

Q17. Is fear or threat used as a rhetorical device?
  Options [single]: Yes | Somewhat | No

Q18. Are calls to action present?
  Options [single]: Yes | No

Q18a. Types of calls to action (ONLY if Q18=Yes; else return empty list)
  Options [multi-select]: Calls for audience to like the content | 
  Calls for audience to share the content | 
  Calls for audience to follow the speaker on social media | 
  Calls for men to follow more traditional gender norms | 
  Calls for men to follow more equitable gender norms | 
  Calls for women to follow more traditional gender norms | 
  Calls for women to follow more equitable gender norms | 
  Calls for politicians or social figures to do something | 
  Calls for audience to vote in a different way | Other

Q18b. If Q18a includes "Other", describe the other call to action. Else empty string.

=== REQUIRED JSON RESPONSE FORMAT ===
Return ONLY valid JSON with these exact keys (no extra keys, no markdown):
{{
  "q1": "Yes" or "No",
  "q1a": [list of strings],
  "q1b": "string",
  "q2": [list of strings],
  "q2a": "string",
  "q3": "string",
  "q3a": "string",
  "q4": "string",
  "q5": "string",
  "q6": "string",
  "q7": [list of strings],
  "q7a": "string",
  "q8": [list of strings],
  "q8a": "string",
  "q9": [list of strings],
  "q9a": "string",
  "q10": [list of strings],
  "q10a": "string",
  "q11": [list of strings],
  "q12": "string",
  "q12a": "string",
  "q13": "string",
  "q13a": "string",
  "q14": "string",
  "q15": "string",
  "q16": "string",
  "q17": "string",
  "q18": "string",
  "q18a": [list of strings],
  "q18b": "string"
}}"""

print("Prompt builder defined.")


In [ ]:
# ── Cell 6: Valid option sets (for validation) ─────────────────────────────────

VALID = {
    "q1":   {"Yes", "No"},
    "q1a":  {"Compelling question", "Use of all CAPS", "Humor or sarcasm",
             "Shares something violent or gross", "Shares something sexual",
             "Shares something surprising",
             "Uses a news headline or social media trend as opener",
             "Interesting visual or meme", "Other"},
    "q2":   {"Dating/marriage", "Friends/socializing", "Family/children",
             "Money/status", "Fitness/self-improvement", "Mental health",
             "Gender issues, e.g. equality", "Social issues, e.g. corruption",
             "Religion/morality", "Gaming/technology", "Other"},
    "q3":   {"Interview/conversational content", "Motivational/self-help content",
             "Commentary/reaction content", "Other"},
    "q4":   {"Yes, explicitly", "Yes, implicitly", "No"},
    "q5":   {"More regressive/traditional/restrictive", "More progressive/equitable/expansive",
             "Mixed/unclear", "Does not address masculinity or gender norms"},
    "q6":   {"Yes", "No", "Unclear", "Not applicable"},
    "q7":   {"Men need to dominate/lead", "Men need to provide/succeed",
             "Men are disadvantaged/victims", "Men need to improve themselves",
             "Men need to be fully self-reliant", "Men need to be emotionally open",
             "Men need to not show emotions", "Men need to be equal partners",
             "Mixed/unclear", "Other", "Not applicable"},
    "q8":   {"Kenyan or Nigerian political/social problems",
             "Global political/social/cultural problems",
             "Western political/social influence", "Women/feminism",
             "Men's behavior", "Economic/status pressure",
             "Mental health/emotional struggle",
             "No clear problem is identified", "Other"},
    "q9":   {"Social or political change", "Assert dominance/control",
             "More wealth/status", "More self-discipline/fitness",
             "More emotional growth/healing", "More equality/respect for men",
             "More equality/respect for women", "Building community",
             "No clear solution", "Other"},
    "q10":  {"Advice/instruction", "Personal story", "Commentary/opinion",
             "Debate/argument", "Humor/satire", "Motivational speech",
             "News/telling facts", "Other"},
    "q11":  {"Entertainment/escapism", "Information seeking",
             "Connection/social interaction", "Self expression/identity construction",
             "Status seeking", "Documentation of events", "None of these apply"},
    "q12":  {"Generalizations about men/women", "Personal experience",
             "Stories about men/women", "Cultural/social observations",
             "Facts/statistics", "Moral/religious claims", "Mixed",
             "No support", "Other"},
    "q13":  {"No justification", "Anecdotal examples", "Presented as common sense",
             "References data", "References religion/tradition",
             "References external sources, such as other influencers", "Other"},
    "q14":  {"Negative", "Positive", "Mixed", "Neutral", "Unclear", "Not mentioned"},
    "q15":  {"Negative", "Positive", "Mixed", "Neutral", "Unclear", "Not mentioned"},
    "q16":  {"Negative", "Positive", "Mixed", "Neutral", "Unclear", "Not mentioned"},
    "q17":  {"Yes", "Somewhat", "No"},
    "q18":  {"Yes", "No"},
    "q18a": {"Calls for audience to like the content",
             "Calls for audience to share the content",
             "Calls for audience to follow the speaker on social media",
             "Calls for men to follow more traditional gender norms",
             "Calls for men to follow more equitable gender norms",
             "Calls for women to follow more traditional gender norms",
             "Calls for women to follow more equitable gender norms",
             "Calls for politicians or social figures to do something",
             "Calls for audience to vote in a different way", "Other"},
}

# Multi-select question keys
MULTI_SELECT_KEYS = {"q1a", "q2", "q7", "q8", "q9", "q10", "q11", "q18a"}

print("Valid option sets defined.")


In [ ]:
# ── Cell 7: Post-processing & conditional logic enforcement ────────────────────

def clean_list_field(val, valid_set):
    """Ensure a list field only contains valid options."""
    if not isinstance(val, list):
        val = []
    return [v for v in val if v in valid_set]

def clean_single_field(val, valid_set, default=""):
    """Ensure a single-select field is a valid option."""
    if val in valid_set:
        return val
    return default

def enforce_other_detail(text: str, label: str) -> str:
    t = str(text or "").strip()
    if len(t.split()) >= 5:
        return t
    return f"Other ({label}): specific nuance not covered by listed options."

def enforce_content_conditionals(r: dict) -> dict:
    """Apply ALL conditional logic rules in post-processing.
    r is a dict with keys q1..q18b.
    Returns cleaned dict.
    """
    # ── Validate/clean all fields against valid option sets ────────────────────
    r["q1"]  = clean_single_field(r.get("q1", ""),  VALID["q1"],  "No")
    r["q1a"] = clean_list_field(r.get("q1a", []),   VALID["q1a"])
    r["q1b"] = str(r.get("q1b", "") or "")

    r["q2"]  = clean_list_field(r.get("q2", []),    VALID["q2"])
    r["q2a"] = str(r.get("q2a", "") or "")

    r["q3"]  = clean_single_field(r.get("q3", ""),  VALID["q3"],  "Other")
    r["q3a"] = str(r.get("q3a", "") or "")

    r["q4"]  = clean_single_field(r.get("q4", ""),  VALID["q4"],  "No")
    r["q5"]  = clean_single_field(r.get("q5", ""),  VALID["q5"],  "Does not address masculinity or gender norms")
    r["q6"]  = clean_single_field(r.get("q6", ""),  VALID["q6"],  "Not applicable")
    r["q7"]  = clean_list_field(r.get("q7", []),    VALID["q7"])
    r["q7a"] = str(r.get("q7a", "") or "")

    r["q8"]  = clean_list_field(r.get("q8", []),    VALID["q8"])
    r["q8a"] = str(r.get("q8a", "") or "")

    r["q9"]  = clean_list_field(r.get("q9", []),    VALID["q9"])
    r["q9a"] = str(r.get("q9a", "") or "")

    r["q10"]  = clean_list_field(r.get("q10", []),  VALID["q10"])
    r["q10a"] = str(r.get("q10a", "") or "")

    r["q11"] = clean_list_field(r.get("q11", []),   VALID["q11"])

    r["q12"]  = clean_single_field(r.get("q12", ""), VALID["q12"], "No support")
    r["q12a"] = str(r.get("q12a", "") or "")

    r["q13"]  = clean_single_field(r.get("q13", ""), VALID["q13"], "No justification")
    r["q13a"] = str(r.get("q13a", "") or "")

    r["q14"] = clean_single_field(r.get("q14", ""), VALID["q14"], "Unclear")
    r["q15"] = clean_single_field(r.get("q15", ""), VALID["q15"], "Unclear")
    r["q16"] = clean_single_field(r.get("q16", ""), VALID["q16"], "Unclear")
    r["q17"] = clean_single_field(r.get("q17", ""), VALID["q17"], "No")
    r["q18"] = clean_single_field(r.get("q18", ""), VALID["q18"], "No")
    r["q18a"] = clean_list_field(r.get("q18a", []),  VALID["q18a"])
    r["q18b"] = str(r.get("q18b", "") or "")

    # ── Conditional logic enforcement ──────────────────────────────────────────

    # Q1 → Q1a, Q1b
    if r["q1"] == "No":
        r["q1a"] = []
        r["q1b"] = ""
    else:  # Q1=Yes
        if not r["q1a"]:
            # LLM said Yes but gave no strategies — default to Shares something surprising
            r["q1a"] = ["Shares something surprising"]
        if "Other" not in r["q1a"]:
            r["q1b"] = ""
        else:
            r["q1b"] = enforce_other_detail(r["q1b"], "attention strategy")

    # Q2 → Q2a
    if "Other" not in r["q2"]:
        r["q2a"] = ""
    else:
        r["q2a"] = enforce_other_detail(r["q2a"], "primary topic")

    # Q3 → Q3a
    if r["q3"] != "Other":
        r["q3a"] = ""
    else:
        r["q3a"] = enforce_other_detail(r["q3a"], "content type")

    # Q4 → Q5, Q6, Q7
    if r["q4"] == "No":
        r["q5"] = "Does not address masculinity or gender norms"
        r["q6"] = "Not applicable"
        r["q7"] = ["Not applicable"]
        r["q7a"] = ""
    else:
        # Q6 ≠ Yes → Q7 not applicable
        if r["q6"] != "Yes":
            r["q7"] = ["Not applicable"]
            r["q7a"] = ""
        else:
            # Q6=Yes: Q7 must be non-empty (no "Not applicable" unless explicitly set)
            r["q7"] = [v for v in r["q7"] if v != "Not applicable"] or ["Mixed/unclear"]
            if "Other" not in r["q7"]:
                r["q7a"] = ""
            else:
                r["q7a"] = enforce_other_detail(r["q7a"], "directive for men")

    # Q8 → Q8a
    if "Other" not in r["q8"]:
        r["q8a"] = ""
    else:
        r["q8a"] = enforce_other_detail(r["q8a"], "problem identified")

    # Q8 must be non-empty
    if not r["q8"]:
        r["q8"] = ["No clear problem is identified"]

    # Q9 → Q9a
    if "Other" not in r["q9"]:
        r["q9a"] = ""
    else:
        r["q9a"] = enforce_other_detail(r["q9a"], "solution proposed")

    # Q9 must be non-empty
    if not r["q9"]:
        r["q9"] = ["No clear solution"]

    # Q10 → Q10a
    if "Other" not in r["q10"]:
        r["q10a"] = ""
    else:
        r["q10a"] = enforce_other_detail(r["q10a"], "communication mode")

    # Q10 must be non-empty
    if not r["q10"]:
        r["q10"] = ["Commentary/opinion"]

    # Q11 must be non-empty
    if not r["q11"]:
        r["q11"] = ["None of these apply"]

    # Q12 → Q12a
    if r["q12"] != "Other":
        r["q12a"] = ""
    else:
        r["q12a"] = enforce_other_detail(r["q12a"], "claim support")

    # Q13 → Q13a
    if r["q13"] != "Other":
        r["q13a"] = ""
    else:
        r["q13a"] = enforce_other_detail(r["q13a"], "claim justification")

    # Q18 → Q18a, Q18b
    if r["q18"] == "No":
        r["q18a"] = []
        r["q18b"] = ""
    else:  # Q18=Yes
        if not r["q18a"]:
            r["q18a"] = ["Calls for audience to follow the speaker on social media"]
        if "Other" not in r["q18a"]:
            r["q18b"] = ""
        else:
            r["q18b"] = enforce_other_detail(r["q18b"], "call to action")

    return r


def row_to_output(content_id: str, context: str, text: str, r: dict) -> dict:
    """Convert cleaned coding dict to output row dict with EXACT column names."""
    def join_list(lst):
        return ", ".join(lst) if lst else ""

    return {
        "Content ID":                               content_id,
        "Context":                                  context,
        "Content Text / Description":               text,
        "Q1. Attention-getter (Yes/No)":             r["q1"],
        "Q1a. Attention-getting strategies":         join_list(r["q1a"]),
        "Q1b. Other strategy":                       r["q1b"],
        "Q2. Primary topic(s)":                      join_list(r["q2"]),
        "Q2a. Other topic":                          r["q2a"],
        "Q3. Type of content":                       r["q3"],
        "Q3a. Other content type":                   r["q3a"],
        "Q4. Addresses masculinity / gender norms":  r["q4"],
        "Q5. Type of masculinity / gender norms":    r["q5"],
        "Q6. Addresses what men should do":          r["q6"],
        "Q7. What men do or need to do":             join_list(r["q7"]),
        "Q7a. Other directive":                      r["q7a"],
        "Q8. Problem identified":                    join_list(r["q8"]),
        "Q8a. Other problem":                        r["q8a"],
        "Q9. Solution proposed":                     join_list(r["q9"]),
        "Q9a. Other solution":                       r["q9a"],
        "Q10. Communication mode":                   join_list(r["q10"]),
        "Q10a. Other communication mode":            r["q10a"],
        "Q11. Audience needs":                       join_list(r["q11"]),
        "Q12. How claims are supported":             r["q12"],
        "Q12a. Other claim support":                 r["q12a"],
        "Q13. How claims are justified":             r["q13"],
        "Q13a. Other justification":                 r["q13a"],
        "Q14. Sentiment toward men":                 r["q14"],
        "Q15. Sentiment toward women":               r["q15"],
        "Q16. Sentiment toward traditional gender norms": r["q16"],
        "Q17. Fear or threat used":                  r["q17"],
        "Q18. Calls to action present":              r["q18"],
        "Q18a. Types of calls to action":            join_list(r["q18a"]),
        "Q18b. Other call to action":                r["q18b"],
    }

print("Post-processing functions defined.")


In [ ]:
# ── Cell 8: Async API caller ───────────────────────────────────────────────────

async def call_llm_content(content_id: str, country: str, context: str, text: str) -> dict:
    """Call GPT-4o-mini for one content item. Returns cleaned coding dict."""
    prompt = build_content_prompt(content_id, country, context, text)
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            response = await client.chat.completions.create(
                model=MODEL,
                seed=SEED,
                temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt},
                ],
            )
            raw = response.choices[0].message.content
            parsed = json.loads(raw)
            cleaned = enforce_content_conditionals(parsed)
            return cleaned
        except json.JSONDecodeError as e:
            last_exc = e
            print(f"  JSON decode error for {content_id} (attempt {attempt+1}): {e}")
            await asyncio.sleep(2 ** attempt)
        except Exception as e:
            last_exc = e
            wait = 2 ** attempt
            print(f"  API error for {content_id} (attempt {attempt+1}): {e} — retrying in {wait}s")
            await asyncio.sleep(wait)
    raise RuntimeError(f"Failed to code {content_id} after {MAX_RETRIES} attempts: {last_exc}")


async def code_country_content(df: pd.DataFrame, country: str,
                                cache_df: pd.DataFrame, cached_keys: set) -> list:
    """Code all rows for one country. Returns list of output row dicts."""
    results = []
    # Pull cached results for this country
    if len(cache_df) > 0 and "_cache_key" in cache_df.columns:
        country_cache = cache_df[cache_df["_country"] == country].copy()
    else:
        country_cache = pd.DataFrame()

    # Build queue of uncached rows
    to_code = []
    for _, row in df.iterrows():
        cid = str(row["Content ID"])
        key = cache_key(cid, country)
        if key in cached_keys:
            # Pull from cache
            cached_row = country_cache[country_cache["_cache_key"] == key]
            if len(cached_row) > 0:
                out_row = json.loads(cached_row.iloc[0]["_result_json"])
                results.append((cid, out_row))
                continue
        to_code.append(row)

    print(f"{country}: {len(results)} cached, {len(to_code)} to code")

    if not to_code:
        # Return results in original order
        ordered = []
        for _, row in df.iterrows():
            cid = str(row["Content ID"])
            for r_cid, r_data in results:
                if r_cid == cid:
                    ordered.append(row_to_output(cid, str(row["Context"]),
                                                 str(row["Content Text / Description"]), r_data))
                    break
        return ordered

    # Semaphore for concurrency
    sem = asyncio.Semaphore(CONCURRENCY)
    new_cache_rows = []

    async def code_one(row):
        async with sem:
            cid     = str(row["Content ID"])
            ctx     = str(row["Context"])
            txt     = str(row["Content Text / Description"])
            coded   = await call_llm_content(cid, country, ctx, txt)
            key     = cache_key(cid, country)
            return cid, ctx, txt, coded, key

    # Process in batches of BATCH_SAVE for periodic cache saves
    nonlocal_cache_df = [cache_df.copy()]
    nonlocal_cached_keys = [set(cached_keys)]

    batch_results = []
    tasks = [code_one(row) for row in to_code]

    completed = 0
    for coro in asyncio.as_completed(tasks):
        cid, ctx, txt, coded, key = await coro
        results.append((cid, coded))
        new_cache_rows.append({
            "_cache_key":  key,
            "_country":    country,
            "_result_json": json.dumps(coded),
        })
        completed += 1
        print(f"  [{country}] Coded {completed}/{len(to_code)}: {cid}")

        # Periodic cache save
        if completed % BATCH_SAVE == 0:
            new_rows_df = pd.DataFrame(new_cache_rows)
            updated_cache = pd.concat([nonlocal_cache_df[0], new_rows_df], ignore_index=True)
            updated_cache = updated_cache.drop_duplicates(subset=["_cache_key"], keep="last")
            save_cache(updated_cache)
            nonlocal_cache_df[0] = updated_cache
            print(f"  Cache saved ({len(updated_cache)} total rows)")

    # Final cache save
    if new_cache_rows:
        new_rows_df = pd.DataFrame(new_cache_rows)
        updated_cache = pd.concat([nonlocal_cache_df[0], new_rows_df], ignore_index=True)
        updated_cache = updated_cache.drop_duplicates(subset=["_cache_key"], keep="last")
        save_cache(updated_cache)
        print(f"  Final cache saved ({len(updated_cache)} total rows)")
        # Update outer cache_df reference for next country call
        nonlocal_cache_df[0] = updated_cache

    # Build final ordered output list
    result_map = {cid: coded for cid, coded in results}
    ordered = []
    for _, row in df.iterrows():
        cid = str(row["Content ID"])
        coded = result_map[cid]
        ordered.append(row_to_output(cid, str(row["Context"]),
                                      str(row["Content Text / Description"]), coded))
    return ordered

print("Async coding functions defined.")


In [ ]:
# ── Cell 9: Run coding (Nigeria then Kenya) ────────────────────────────────────

print("=" * 60)
print("STARTING CONTENT ANALYSIS CODING")
print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

# Reload cache fresh
cache_df = load_cache()
cached_keys = set(cache_df["_cache_key"].tolist()) if "_cache_key" in cache_df.columns else set()

print(f"\nCoding Nigeria ({len(nigeria_df)} rows)...")
t0 = time.time()
nigeria_results = await code_country_content(nigeria_df, "Nigeria", cache_df, cached_keys)
print(f"Nigeria done in {time.time()-t0:.1f}s")

# Reload cache after Nigeria (may have been updated)
cache_df = load_cache()
cached_keys = set(cache_df["_cache_key"].tolist()) if "_cache_key" in cache_df.columns else set()

print(f"\nCoding Kenya ({len(kenya_df)} rows)...")
t0 = time.time()
kenya_results = await code_country_content(kenya_df, "Kenya", cache_df, cached_keys)
print(f"Kenya done in {time.time()-t0:.1f}s")

# Convert to DataFrames
nigeria_out = pd.DataFrame(nigeria_results, columns=OUTPUT_COLUMNS)
kenya_out   = pd.DataFrame(kenya_results,   columns=OUTPUT_COLUMNS)

assert len(nigeria_out) == 200, f"Nigeria output: expected 200, got {len(nigeria_out)}"
assert len(kenya_out)   == 200, f"Kenya output: expected 200, got {len(kenya_out)}"

print(f"\nNigeria output rows: {len(nigeria_out)}")
print(f"Kenya output rows:   {len(kenya_out)}")
print("\nRow count assertions passed.")


In [ ]:
# ── Cell 10: Validation report ─────────────────────────────────────────────────

def validate_content_output(df: pd.DataFrame, country: str):
    """Print a full validation report for one country's output."""
    print(f"\n{'='*60}")
    print(f"VALIDATION REPORT — {country} ({len(df)} rows)")
    print(f"{'='*60}")

    violations = []

    for i, row in df.iterrows():
        cid = row["Content ID"]

        q1  = row["Q1. Attention-getter (Yes/No)"]
        q1a = row["Q1a. Attention-getting strategies"]
        q1b = row["Q1b. Other strategy"]
        q2  = row["Q2. Primary topic(s)"]
        q2a = row["Q2a. Other topic"]
        q3  = row["Q3. Type of content"]
        q3a = row["Q3a. Other content type"]
        q4  = row["Q4. Addresses masculinity / gender norms"]
        q5  = row["Q5. Type of masculinity / gender norms"]
        q6  = row["Q6. Addresses what men should do"]
        q7  = row["Q7. What men do or need to do"]
        q7a = row["Q7a. Other directive"]
        q8  = row["Q8. Problem identified"]
        q8a = row["Q8a. Other problem"]
        q9  = row["Q9. Solution proposed"]
        q9a = row["Q9a. Other solution"]
        q10 = row["Q10. Communication mode"]
        q10a= row["Q10a. Other communication mode"]
        q12 = row["Q12. How claims are supported"]
        q12a= row["Q12a. Other claim support"]
        q13 = row["Q13. How claims are justified"]
        q13a= row["Q13a. Other justification"]
        q18 = row["Q18. Calls to action present"]
        q18a= row["Q18a. Types of calls to action"]
        q18b= row["Q18b. Other call to action"]

        def viol(msg):
            violations.append(f"  [{cid}] {msg}")

        # Q1 conditionals
        if q1 == "No" and q1a:
            viol(f"Q1=No but Q1a non-empty: {q1a!r}")
        if q1 == "No" and q1b:
            viol(f"Q1=No but Q1b non-empty: {q1b!r}")
        if q1 == "Yes" and not q1a:
            viol("Q1=Yes but Q1a empty")
        if q1a and "Other" not in q1a and q1b:
            viol(f"Q1a no Other but Q1b non-empty: {q1b!r}")
        if q1a and "Other" in q1a and not q1b:
            viol("Q1a includes Other but Q1b empty")

        # Q2 conditionals
        if "Other" not in q2 and q2a:
            viol(f"Q2 no Other but Q2a non-empty: {q2a!r}")
        if "Other" in q2 and not q2a:
            viol("Q2 includes Other but Q2a empty")

        # Q3 conditionals
        if q3 != "Other" and q3a:
            viol(f"Q3≠Other but Q3a non-empty: {q3a!r}")
        if q3 == "Other" and not q3a:
            viol("Q3=Other but Q3a empty")

        # Q4 conditionals
        if q4 == "No":
            if q5 != "Does not address masculinity or gender norms":
                viol(f"Q4=No but Q5={q5!r}")
            if q6 != "Not applicable":
                viol(f"Q4=No but Q6={q6!r}")
            if q7 not in ["", "Not applicable"]:
                viol(f"Q4=No but Q7={q7!r}")

        if q4 != "No" and q6 != "Yes":
            if q7 not in ["", "Not applicable"]:
                viol(f"Q6≠Yes but Q7={q7!r}")

        # Q7 conditionals
        if "Other" not in q7 and q7a:
            viol(f"Q7 no Other but Q7a non-empty: {q7a!r}")
        if "Other" in q7 and not q7a:
            viol("Q7 includes Other but Q7a empty")

        # Q8, Q9, Q10
        if "Other" not in q8 and q8a:
            viol(f"Q8 no Other but Q8a non-empty: {q8a!r}")
        if "Other" in q8 and not q8a:
            viol("Q8 includes Other but Q8a empty")
        if "Other" not in q9 and q9a:
            viol(f"Q9 no Other but Q9a non-empty: {q9a!r}")
        if "Other" in q9 and not q9a:
            viol("Q9 includes Other but Q9a empty")
        if "Other" not in q10 and q10a:
            viol(f"Q10 no Other but Q10a non-empty: {q10a!r}")
        if "Other" in q10 and not q10a:
            viol("Q10 includes Other but Q10a empty")

        # Q12, Q13
        if q12 != "Other" and q12a:
            viol(f"Q12≠Other but Q12a non-empty: {q12a!r}")
        if q12 == "Other" and not q12a:
            viol("Q12=Other but Q12a empty")
        if q13 != "Other" and q13a:
            viol(f"Q13≠Other but Q13a non-empty: {q13a!r}")
        if q13 == "Other" and not q13a:
            viol("Q13=Other but Q13a empty")

        # Q18 conditionals
        if q18 == "No" and q18a:
            viol(f"Q18=No but Q18a non-empty: {q18a!r}")
        if q18 == "No" and q18b:
            viol(f"Q18=No but Q18b non-empty: {q18b!r}")
        if q18 == "Yes" and not q18a:
            viol("Q18=Yes but Q18a empty")
        if q18a and "Other" not in q18a and q18b:
            viol(f"Q18a no Other but Q18b non-empty: {q18b!r}")
        if q18a and "Other" in q18a and not q18b:
            viol("Q18a includes Other but Q18b empty")

    if violations:
        print(f"VIOLATIONS ({len(violations)}):")
        for v in violations:
            print(v)
    else:
        print("No conditional logic violations found.")

    print(f"\n--- Key field distributions ---")
    for col in ["Q1. Attention-getter (Yes/No)",
                "Q3. Type of content",
                "Q4. Addresses masculinity / gender norms",
                "Q5. Type of masculinity / gender norms",
                "Q6. Addresses what men should do",
                "Q17. Fear or threat used",
                "Q18. Calls to action present"]:
        print(f"\n{col}:")
        print(df[col].value_counts().to_string())

    return len(violations)

n_viol_ng = validate_content_output(nigeria_out, "Nigeria")
n_viol_ke = validate_content_output(kenya_out, "Kenya")
print(f"\nTotal violations: Nigeria={n_viol_ng}, Kenya={n_viol_ke}")


In [ ]:
# ── Cell 11: Write Excel output ────────────────────────────────────────────────

HEADER_FILL  = PatternFill("solid", fgColor="1F3864")  # dark navy
HEADER_FONT  = Font(bold=True, color="FFFFFF", size=11)
HEADER_ALIGN = Alignment(horizontal="center", vertical="center", wrap_text=True)
ALT_FILL     = PatternFill("solid", fgColor="EBF0FA")


def write_data_sheet(ws, df: pd.DataFrame, country: str):
    """Write a country's DataFrame to a worksheet with formatting."""
    ws.title = f"{country} - LLM Coding"

    # Write headers
    for col_idx, col_name in enumerate(df.columns, 1):
        cell = ws.cell(row=1, column=col_idx, value=col_name)
        cell.font  = HEADER_FONT
        cell.fill  = HEADER_FILL
        cell.alignment = HEADER_ALIGN

    # Write data rows
    for row_idx, (_, row) in enumerate(df.iterrows(), 2):
        fill = ALT_FILL if row_idx % 2 == 0 else None
        for col_idx, val in enumerate(row, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=val)
            cell.alignment = Alignment(wrap_text=True, vertical="top")
            if fill:
                cell.fill = fill

    # Freeze top row
    ws.freeze_panes = "A2"

    # Auto-width columns (capped)
    for col_idx, col_name in enumerate(df.columns, 1):
        col_letter = get_column_letter(col_idx)
        # Measure max content width
        max_len = len(str(col_name))
        for _, row in df.iterrows():
            val = str(row.iloc[col_idx - 1])
            # Use first line length for wrapped text
            first_line = val.split("\n")[0]
            max_len = max(max_len, min(len(first_line), 80))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 80)

    # Set row heights
    ws.row_dimensions[1].height = 40
    for row_idx in range(2, len(df) + 2):
        ws.row_dimensions[row_idx].height = 60


def write_methodology_sheet(ws, run_date: str):
    """Write a methodology description sheet."""
    ws.title = "Methodology"
    rows = [
        ["LLM Content Analysis — Methodology"],
        [],
        ["Field", "Value"],
        ["Model", MODEL],
        ["Seed", str(SEED)],
        ["Temperature", "0"],
        ["Concurrency", str(CONCURRENCY)],
        ["Date run", run_date],
        ["Nigeria source rows", "200 (from 'Nigeria — All Items' sheet)"],
        ["Kenya source rows", "200 (from 'Kenya — All Items' sheet)"],
        ["Source file", str(CONTENT_MASTER)],
        [],
        ["Approach"],
        ["The LLM received only the Content ID, Context, and Content Text / Description for each item."],
        ["Human coder answers were NOT sent to the model. The model coded from scratch."],
        ["All conditional logic was enforced in post-processing after the LLM response."],
        ["Multi-select fields are stored as comma-joined strings matching the human codebook format."],
        ["Results are cached in a parquet file to avoid duplicate API calls on re-runs."],
        [],
        ["Theme Taxonomy (background context provided to LLM — not an output column)"],
        ["1. Authority and Submission — explicit hierarchy, headship, obedience, control"],
        ["2. Male Victimhood — men framed as harmed, exploited, disadvantaged"],
        ["3. Gender Grievance — distrust/resentment/hostility toward women or feminism"],
        ["4. Sexual Morality — cheating, body count, purity, sexual respectability"],
        ["5. Relationship Tactics — tactical dating advice, scarcity, frame, withdrawal"],
        ["6. Provider and Status — money, career, status as proof of manhood"],
        ["7. Male Accountability — men must change, hold others accountable"],
        ["8. Egalitarian Partnership — mutual respect, shared parenting, reciprocity"],
        ["9. Gender-Based Violence and Consent — rape, consent, abuse, femicide, FGM"],
        ["10. Trauma and Mental Health — trauma, depression, healing, therapy"],
        ["11. Self-Discipline and Self-Mastery — discipline, fitness, hustle, personal growth"],
        ["12. Marriage and Family — marriage, divorce, fatherhood, parenting"],
        ["13. Faith and Moral Repair — God, prayer, spiritual healing tied to masculinity"],
        ["Unclear — uncodable, off-topic, low-signal"],
    ]
    for r_idx, row_data in enumerate(rows, 1):
        for c_idx, val in enumerate(row_data, 1):
            cell = ws.cell(row=r_idx, column=c_idx, value=val)
            if r_idx == 1 or (r_idx == 3 and c_idx <= 2) or (r_idx == 13 and c_idx == 1) or (r_idx == 20 and c_idx == 1):
                cell.font = Font(bold=True, size=12 if r_idx == 1 else 11)
            cell.alignment = Alignment(wrap_text=True)
    ws.column_dimensions["A"].width = 40
    ws.column_dimensions["B"].width = 80


# ── Build workbook ─────────────────────────────────────────────────────────────
run_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
wb = openpyxl.Workbook()

# Nigeria sheet
ws_ng = wb.active
write_data_sheet(ws_ng, nigeria_out, "Nigeria")

# Kenya sheet
ws_ke = wb.create_sheet()
write_data_sheet(ws_ke, kenya_out, "Kenya")

# Methodology sheet
ws_meth = wb.create_sheet()
write_methodology_sheet(ws_meth, run_date)

# Save
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
wb.save(OUTPUT_PATH)
print(f"\nOutput saved to: {OUTPUT_PATH}")
print(f"Nigeria sheet: {len(nigeria_out)} rows")
print(f"Kenya sheet:   {len(kenya_out)} rows")
print(f"Sheets: {[ws.title for ws in wb.worksheets]}")
print(f"\nDone at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


# Also save country-specific files
COUNTRY_OUTPUTS = {
    "Nigeria": OUTPUT_PATH.parent / "LLM Coding - Content Analysis - Nigeria.xlsx",
    "Kenya": OUTPUT_PATH.parent / "LLM Coding - Content Analysis - Kenya.xlsx",
}

def write_single_country_file(path: Path, df: pd.DataFrame, country: str, run_date: str):
    wb_single = openpyxl.Workbook()
    ws = wb_single.active
    write_data_sheet(ws, df, country)
    ws_meth = wb_single.create_sheet()
    write_methodology_sheet(ws_meth, run_date)
    wb_single.save(path)

write_single_country_file(COUNTRY_OUTPUTS["Nigeria"], nigeria_out, "Nigeria", run_date)
write_single_country_file(COUNTRY_OUTPUTS["Kenya"], kenya_out, "Kenya", run_date)
